Circos visualization
================

In this vignette, you can learn how to visualize the output of a
NicheNet analysis in a circos plot (also called a chord diagram) via the
`pycirclize` package. This vignette follows the same workflow as shown in
[Perform NicheNet analysis starting from an AnnData
object](wrapper.ipynb).

This tutorial was made upon popular request to demonstrate how those two
tutorials can be combined into one analysis workflow. Note that we as
developers of NicheNet generally recommend a visualization of the output
by combining several heatmaps (ligand activity, ligand-target links,
ligand-receptor links, ligand expression, ligand LFC,…) over using a
circos plot visualization. This is especially true for cases with many
sender cell types and ligands that are expressed by more than one sender
cell type. Because in those cases, the circos plot is much less
informative and could lead to wrong interpretation of the results.

We will again use the NICHE-seq data from Medaglia et al. (2017), which
profiles several immune cell types in the T cell area in the inguinal
lymph node before and 72 hours after lymphocytic choriomeningitis virus
(LCMV) infection. 

## Prepare NicheNet analysis

In [1]:
from nichenetpy.gene_symbol import mouse_alias_info
from nichenetpy.wrappers import run_nichenet
from nichenetpy.visualization import assign_ligands_to_celltype

import anndata
import os
import requests
import pickle

Download the model files

In [2]:
model_path = os.path.normpath("./tutorial_files/model/mouse")
if not os.path.exists(model_path):
    os.makedirs(model_path)
filename = "nichenet_mouse.pkl"
file_path = os.path.join("./tutorial_files", filename)
if not os.path.exists(file_path):
    res = requests.get(f"https://zenodo.org/records/14887637/files/{filename}")
    with open(file_path, "wb") as file:
        file.write(res.content)

Download the AnnData object

In [3]:
data_path = os.path.normpath("./tutorial_files/AnnData")
if not os.path.exists(data_path):
    os.makedirs(data_path)
filename = "annData3531889.h5"
file_path = os.path.join(data_path, filename)
if not os.path.exists(file_path):
    res = requests.get(f"https://zenodo.org/records/14859451/files/{filename}")
    with open(file_path, "wb") as file:
        file.write(res.content)

### Read in the expression data of interacting cells

In [4]:
ann = anndata.io.read_h5ad(os.path.join(data_path, "annData3531889.h5"))
ann.var_names = ann.var["gene"]
mouse_alias_info.alias_to_symbol(ann)

## Read in NicheNet's networks

The ligand-target prior model is a matrix describing the potential that a ligand may regulate a target gene, and it is used to run the ligand activity analysis. The ligand-receptor network contains information on potential ligand-receptor bindings, and it is used to identify potential ligands. 

In [5]:
with open("./tutorial_files/nichenet_mouse.pkl", "rb") as file:
    model = pickle.loads(file.read())
predictor = model["predictor"]
lr_network = model["lr_network"]
lr_sig = model["lr_sig"]

## Perform the NicheNet analysis

In [6]:
sender_celltypes = ("CD4 T", "Treg", "Mono", "NK", "B", "DC")
res = run_nichenet(
    ann,
    predictor,
    lr_network,
    "CD8 T",
    "aggregate",
    "LCMV",
    "SS",
    sender_celltypes=sender_celltypes,
    lr_sig=lr_sig
)
best_upstream_ligands = res["best_upstream_ligands"]

## Circos plots to visualize ligand-target and ligand-receptor interactions

This visualization groups the top predicted active ligands according to
the strongest expressing cell type. Therefore we need to determine per
cell type which ligands they express more strongly than the other cell
types.

### Assign ligands to sender cells

To assign ligands to sender cell type, we can look for which sender cell
types show a mean expression that is higher than the mean + one standard
deviation. 

In [7]:
# the best upstream ligands are not necessarily present in the AnnData object
assign_ligands_to_celltype(ann, sorted(set(best_upstream_ligands).intersection(ann.var_names)))

[ 0.27617381 17.29691767  1.30618899  0.3005334   0.11947206  0.19654459
 14.63677136  0.33004932  0.57315346]


{'B': ['Il27'],
 'CD4 T': ['Ebi3'],
 'CD8 T': ['Ebi3'],
 'DC': ['Osm'],
 'Mono': ['Il27'],
 'NK': ['H2-M3', 'Osm'],
 'Treg': ['Osm']}